# Qwen3-VL Pure-VLM - Kaggle
Qwen3-VL detects (and optionally explains) objects directly - no YOLO (RQ1 / RQ2).

**`RUN_MODE`** switches the single inference call between:
- `'detect'`          → bounding boxes only (faster - for the RQ1 detection comparison)
- `'detect+explain'`  → boxes + a one-sentence explanation per object (for RQ2)

Performance fixes (same lessons as the hybrid notebook):
- **fp16 on T4** (bf16 is emulated/slow on Turing; bf16 auto on Ampere+).
- **No 4-bit quantization** - quantizing the vision tower wrecks localization and washes the image to gray.
- `device_map={'': 0}` (no cross-GPU sharding) + `attn_implementation='sdpa'`.
- Unified `apply_chat_template(tokenize=True, return_dict=True)` path (not the legacy two-step processor).
- Greedy decoding with **no** `repetition_penalty` / `no_repeat_ngram_size` (those are CPU logits processors that scan the ~1000 image tokens every step and starve the GPU).

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers accelerate pillow

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HUGGINGFACE_TOKEN')
print('Token loaded.')

In [ ]:
# ── Config + run mode ────────────────────────────────────────────────
import glob
from pathlib import Path
from collections import Counter

# Best-guess Kaggle mount for the attached VisDrone dataset; auto-discovered below if wrong.
IMAGES_DIR = '/kaggle/input/datasets/deividasbuinevicius/visdrone-dataset/VisDrone2019-DET-val/images'

MODEL_ID = 'Qwen/Qwen3-VL-2B-Instruct'

# ── Mode switch ──────────────────────────────────────────────────────
# 'detect'         → boxes only          (RQ1 detection metrics)
# 'detect+explain' → boxes + explanation (RQ2 qualitative)
RUN_MODE = 'detect+explain'

MAX_IMAGES         = 0       # 0 = all 548 val images
MAX_DETS_PER_IMAGE = 15      # explain only the top-15 objects; ALL objects still detected
MAX_PIXELS         = 1280 * 1280   # input resolution cap; higher = better small-object recall but slower
MAX_NEW_TOKENS     = 1536 if RUN_MODE == 'detect' else 2048   # dense aerial scenes need a long JSON array
OUTPUT_DIR         = '/kaggle/working'

# Auto-discover the VisDrone val images folder if the configured path is empty
if not (Path(IMAGES_DIR).is_dir() and list(Path(IMAGES_DIR).glob('*.jpg'))):
    hits = glob.glob('/kaggle/input/**/VisDrone2019-DET-val/images', recursive=True)
    if hits:
        IMAGES_DIR = hits[0]
        print(f'Using discovered val folder: {IMAGES_DIR}')
    else:
        # fallback: the folder under /kaggle/input holding the most .jpg files
        dirs = Counter(str(Path(p).parent) for p in glob.glob('/kaggle/input/**/*.jpg', recursive=True))
        if dirs:
            IMAGES_DIR = dirs.most_common(1)[0][0]
            print(f'Val folder not found by name; using the largest image folder: {IMAGES_DIR}')

n_imgs = len(list(Path(IMAGES_DIR).glob('*.jpg')))
print('Images dir exists:', Path(IMAGES_DIR).is_dir())
print('Images found     :', n_imgs, '(expected 548 for VisDrone2019-DET-val)')
print('Run mode         :', RUN_MODE)
print('max_new_tokens   :', MAX_NEW_TOKENS)
assert n_imgs > 0, 'No .jpg images found - attach the VisDrone dataset or fix IMAGES_DIR.'

In [ ]:
# ── Load Qwen3-VL ──────────────────────────────────────────
# fp16 on T4 (Turing has no bf16 tensor cores); bf16 auto on Ampere+. No 4-bit
# quant - it cripples localization and grays the image. is_bf16_supported() is
# True on T4 (emulated), so gate on compute capability instead.
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

_cc = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
DTYPE = torch.bfloat16 if _cc >= 8 else torch.float16
print(f'GPU compute capability {_cc}.x -> using dtype: {DTYPE}')

processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=MAX_PIXELS)
model     = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map={'': 0},
    attn_implementation='sdpa',
).eval()
print('Loaded ->',
      'device:', next(model.parameters()).device,
      '| dtype:', next(model.parameters()).dtype,
      '| VRAM:', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')

In [ ]:
# ── Helpers: classes, prompts, inference, JSON parser ──────────────
import json, re
from PIL import Image

VISDRONE_CLASSES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van',
    'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor'
]
CLASS_TO_ID = {c: i for i, c in enumerate(VISDRONE_CLASSES)}


def build_detection_prompt():
    """Detection only - class + 0-1000 normalised box."""
    classes = ', '.join(VISDRONE_CLASSES)
    return (
        f"This is an aerial drone image taken from above.\n\n"
        f"Detect all visible objects from these categories: {classes}.\n\n"
        f"For each object you can see, output a JSON object with:\n"
        f"  - class: one of the category names above\n"
        f"  - bbox: [x1, y1, x2, y2] normalised to 0-1000 (0,0 = top-left, 1000,1000 = bottom-right)\n\n"
        f"Return ONLY a valid JSON array. If no objects are visible, return [].\n"
        f"Example: [{{\"class\": \"car\", \"bbox\": [120, 45, 200, 90]}}]"
    )


def build_combined_prompt():
    """Detection + one-sentence explanation per object."""
    classes = ', '.join(VISDRONE_CLASSES)
    return (
        f"This is an aerial drone image taken from above.\n\n"
        f"Detect all visible objects from these categories: {classes}.\n\n"
        f"For each object you can see, output a JSON object with:\n"
        f"  - class: one of the category names above\n"
        f"  - bbox: [x1, y1, x2, y2] normalised to 0-1000 (0,0 = top-left, 1000,1000 = bottom-right)\n"
        f"  - explanation: one sentence describing the object -- its colour, what it looks like, and what it is doing\n\n"
        f"Return ONLY a valid JSON array. If no objects are visible, return [].\n"
        f"Example: [{{\"class\": \"car\", \"bbox\": [120, 45, 200, 90], \"explanation\": \"A white car parked at the roadside.\"}}]"
    )


def get_prompt():
    return build_detection_prompt() if RUN_MODE == 'detect' else build_combined_prompt()


def run_vlm_detection(pil_image, prompt_text, max_new_tokens=MAX_NEW_TOKENS):
    """One inference call; returns the raw response string."""
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': pil_image},
            {'type': 'text',  'text': prompt_text},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors='pt',
    ).to(model.device)

    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    trimmed  = [g[len(i):] for i, g in zip(inputs.input_ids, out)]
    response = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

    del inputs, out, trimmed
    torch.cuda.empty_cache()
    return response


def parse_detections(response, img_w, img_h):
    """Parse the JSON array; tolerate truncated output. Explanation is optional."""
    match = re.search(r'\[.*\]', response, re.DOTALL)
    if not match:
        return [], 'no_json_found'

    raw = match.group()
    # Close any brackets left open by a truncated generation
    raw += '}' * (raw.count('{') - raw.count('}')) + ']' * (raw.count('[') - raw.count(']'))

    try:
        detections = json.loads(raw)
    except json.JSONDecodeError:
        detections = []
        for obj in re.finditer(r'\{[^{}]+\}', raw):
            try:
                detections.append(json.loads(obj.group()))
            except json.JSONDecodeError:
                continue
        if not detections:
            return [], 'json_parse_error'

    results = []
    for det in detections:
        class_name = str(det.get('class', '')).lower().strip()
        if class_name not in CLASS_TO_ID:
            continue
        bbox = det.get('bbox', [])
        if len(bbox) != 4:
            continue
        x1 = max(0, min(float(bbox[0]) / 1000 * img_w, img_w))
        y1 = max(0, min(float(bbox[1]) / 1000 * img_h, img_h))
        x2 = max(0, min(float(bbox[2]) / 1000 * img_w, img_w))
        y2 = max(0, min(float(bbox[3]) / 1000 * img_h, img_h))
        if x2 <= x1 or y2 <= y1:
            continue
        results.append({
            'class_id':    CLASS_TO_ID[class_name],
            'class_name':  class_name,
            'confidence':  1.0,   # VLM gives no meaningful score
            'bbox':        [x1, y1, x2, y2],
            'explanation': str(det.get('explanation', '')).strip(),
        })

    # MAX_DETS_PER_IMAGE caps how many objects are EXPLAINED, not how many are
    # detected: keep every box (for RQ1 detection metrics) and only blank the
    # explanation text beyond the cap.
    if MAX_DETS_PER_IMAGE > 0:
        for _d in results[MAX_DETS_PER_IMAGE:]:
            _d['explanation'] = ''
    return results, 'ok'


print('Helpers defined. Active prompt:',
      'detection only' if RUN_MODE == 'detect' else 'detection + explanation')

In [ ]:
# ── Smoke test (2 images) ───────────────────────────────────
image_paths = sorted(Path(IMAGES_DIR).glob('*.jpg'))[:2]

for image_path in image_paths:
    print(f'\n--- {image_path.name} ---')
    img = Image.open(image_path).convert('RGB')

    response = run_vlm_detection(img, get_prompt())
    print('Raw response (first 500 chars):', response[:500])

    detections, status = parse_detections(response, img.width, img.height)
    print(f'Parse status: {status} | Detections: {len(detections)}')
    for d in detections[:3]:
        bbox_str = [round(v) for v in d['bbox']]
        if RUN_MODE == 'detect+explain':
            print(f"  {d['class_name']:16s} bbox={bbox_str}  ->  {d['explanation']}")
        else:
            print(f"  {d['class_name']:16s} bbox={bbox_str}")

print('\nSmoke test complete.')

In [ ]:
# ── Full run ──────────────────────────────────────────────
import time

all_image_paths = sorted(Path(IMAGES_DIR).glob('*.jpg'))
if MAX_IMAGES and MAX_IMAGES > 0:
    all_image_paths = all_image_paths[:MAX_IMAGES]

print(f'Processing {len(all_image_paths)} images in mode: {RUN_MODE}\n')

prompt       = get_prompt()

# Resume from checkpoint: reload partials so re-running continues instead of
# restarting. Upload qwen3vl_purevlm_partial.json + qwen3vl_timing_partial.json
# to /kaggle/working and set Session > Persistence = "Files only".
import glob as _glob
results_all  = {}
timing_log   = {}
def _find_ckpt(name):
    cand = f'{OUTPUT_DIR}/{name}'
    if Path(cand).is_file():
        return cand
    hits = sorted(_glob.glob(f'/kaggle/input/**/{name}', recursive=True))  # re-uploaded as a dataset
    return hits[0] if hits else None
_partial  = _find_ckpt('qwen3vl_purevlm_partial.json')
_tpartial = _find_ckpt('qwen3vl_timing_partial.json')
if _partial:
    results_all = json.load(open(_partial))
    if _tpartial:
        timing_log = json.load(open(_tpartial))
    print(f'Resuming: {len(results_all)} images already done (from {_partial}) -> will skip them.')
else:
    print('No checkpoint found - starting from scratch.')
predictions  = {
    name: [{'class_id': d['class_id'], 'class_name': d['class_name'],
            'confidence': d.get('confidence', 1.0), 'bbox': d['bbox']} for d in dets]
    for name, dets in results_all.items()
}
parse_errors = 0
run_start    = time.perf_counter()

for i, image_path in enumerate(all_image_paths):
    if image_path.name in results_all:
        continue   # already done in a previous session
    print(f'[{i+1}/{len(all_image_paths)}] {image_path.name}', end=' ... ', flush=True)
    img = Image.open(image_path).convert('RGB')

    t0       = time.perf_counter()
    response = run_vlm_detection(img, prompt)
    inf_time = time.perf_counter() - t0

    detections, status = parse_detections(response, img.width, img.height)
    timing_log[image_path.name] = {'inference_time_s': round(inf_time, 3), 'num_detections': len(detections)}

    if status != 'ok':
        parse_errors += 1
        print(f'parse error ({status}) [{inf_time:.1f}s]')
        results_all[image_path.name] = []
        predictions[image_path.name] = []
        continue

    results_all[image_path.name] = detections
    predictions[image_path.name] = [
        {'class_id': d['class_id'], 'class_name': d['class_name'],
         'confidence': d['confidence'], 'bbox': d['bbox']}
        for d in detections
    ]
    print(f'{len(detections)} detections [{inf_time:.1f}s]')

    if (i + 1) % 10 == 0:
        elapsed = time.perf_counter() - run_start
        eta     = (elapsed / (i + 1)) * (len(all_image_paths) - i - 1)
        print(f'  -- {i+1} done | avg {elapsed/(i+1):.1f}s/img | ETA {eta/60:.0f} min --')
        with open(f'{OUTPUT_DIR}/qwen3vl_purevlm_partial.json', 'w') as f:
            json.dump(results_all, f, indent=2)
        with open(f'{OUTPUT_DIR}/qwen3vl_timing_partial.json', 'w') as f:
            json.dump(timing_log, f, indent=2)

total_elapsed = time.perf_counter() - run_start
times    = [v['inference_time_s'] for v in timing_log.values()]
avg_time = sum(times) / len(times) if times else 0
print(f'\nDone. Parse errors: {parse_errors}/{len(all_image_paths)}')
print(f'Total wall time: {total_elapsed/60:.1f} min | avg {avg_time:.2f}s/img')

In [ ]:
# ── Save  (download from the Output tab) ─────────────────────────
with open(f'{OUTPUT_DIR}/qwen3vl_purevlm_explanations.json', 'w') as f:
    json.dump({
        'model':   MODEL_ID,
        'setting': 'pure-vlm',
        'mode':    RUN_MODE,
        'config':  {'max_images': MAX_IMAGES, 'max_dets_per_image': MAX_DETS_PER_IMAGE,
                    'max_pixels': MAX_PIXELS, 'max_new_tokens': MAX_NEW_TOKENS},
        'images':  results_all,
    }, f, indent=2)

with open(f'{OUTPUT_DIR}/qwen3vl_purevlm_predictions.json', 'w') as f:
    json.dump(predictions, f, indent=2)

with open(f'{OUTPUT_DIR}/qwen3vl_timing.json', 'w') as f:
    json.dump({
        'model': MODEL_ID, 'setting': 'pure-vlm', 'mode': RUN_MODE,
        'note': 'Single inference call per image (detection only, or detection+explanation per RUN_MODE).',
        'summary': {'num_images': len(timing_log),
                    'total_elapsed_min': round(total_elapsed / 60, 2),
                    'avg_inference_time_s': round(avg_time, 3)},
        'per_image': timing_log,
    }, f, indent=2)

total_detections = sum(len(v) for v in results_all.values())
print(f'Mode: {RUN_MODE} | Saved {len(results_all)} images, {total_detections} total detections')
print('→ qwen3vl_purevlm_explanations.json')
print('→ qwen3vl_purevlm_predictions.json')
print('→ qwen3vl_timing.json')